In [4]:
import os
from pathlib import Path
import pandas as pd

In [5]:
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [6]:
df = pd.read_csv("data/combined_clean.csv")
df.shape

(150000, 6)

In [7]:
df.columns

Index(['review_headline', 'review_body', 'star_rating', 'verified_purchase',
       'helpful_votes', 'total_votes'],
      dtype='object')

In [1]:
pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 5.2 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

2026-06-10 14:45:12.468789: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-10 14:45:12.623435: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-10 14:45:12.800075: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-10 14:45:13.008155: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-10 14:45:13.009572: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-10 14:45:13.391309: I tensorflow/core/platform/cpu_feature_guard.cc:

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Sadece problemli yorumları al, embedding için
df_problems = df[df["star_rating"] <= 2]["review_body"].dropna().sample(10000, random_state=42).tolist()
print(f"Toplam: {len(df_problems)} yorum")

Toplam: 10000 yorum


In [9]:
embeddings = model.encode(df_problems, batch_size=64, show_progress_bar=True)
print(embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

(10000, 384)


In [10]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embeddings)

import pandas as pd
df_clustered = pd.DataFrame({
    "review_body": df_problems,
    "cluster": clusters
})

df_clustered["cluster"].value_counts().sort_index()

cluster
0     925
1    2044
2    2507
3    1454
4    1220
5    1850
Name: count, dtype: int64

In [11]:
for i in range(6):
    print(f"\n{'='*40}")
    print(f"CLUSTER {i} ({df_clustered[df_clustered['cluster']==i].shape[0]} yorum)")
    print('='*40)
    samples = df_clustered[df_clustered["cluster"]==i]["review_body"].sample(3, random_state=42).tolist()
    for s in samples:
        print(f"- {s[:200]}")
        print()


CLUSTER 0 (925 yorum)
- This armband worked great for a few days, and then the little velcro cicrles started falling off.  Looking at the backs of them, you can see they were held on by only a minimal amount of glue, to a ba

- If your use of the case in mainly the wallet, and you use your iPod Touch only for music, and the occassional video playback. And you don't go in and out of the waller very often, this case is for you

- didn't fit the ipod and was not able to return. Just a learning lesson on asking more questions before buying.


CLUSTER 1 (2044 yorum)
- Do not recommend.  Product did not work. It was built in a very flimsy fashion and started to come apart right away

- Simple enough to assemble and a decent value but the finish is weak. Sliding a plastic bucket into the bottom section caused scratches. The knob screws were a bit too short to work easily. Had to forc

- This looked real good, the price looked great, and the reviews were also good, so I bought it.  When I tri

0 - Fiziksel hasar, fit sorunu, iade edilemiyor → ürün_kalitesi

1 - Çalışmıyor, kötü yapım, uyumsuzluk → ürün_kalitesi

2 - Kısa sürede bozuldu, çalışmayı bıraktı → ürün_dayanıklılığı

3 - Oyun içeriği beklentiyi karşılamadı → içerik_beklenti

4 - Donanım sorunu, parçalar bozuk → ürün_kalitesi

5 - Bağlantı sorunu, ses kalitesi → performans

In [13]:
cluster_labels = {
    0: "ürün_kalitesi",
    1: "ürün_kalitesi",
    2: "ürün_dayanıklılığı",
    3: "içerik_beklenti",
    4: "ürün_kalitesi",
    5: "performans"
}

df_clustered["problem_category"] = df_clustered["cluster"].map(cluster_labels)
df_clustered["problem_category"].value_counts()

problem_category
ürün_kalitesi         4189
ürün_dayanıklılığı    2507
performans            1850
içerik_beklenti       1454
Name: count, dtype: int64

In [14]:
# df_clustered'ı df ile birleştir
df_labeled = df[df["star_rating"] <= 2].dropna(subset=["review_body"]).copy()
df_labeled = df_labeled[df_labeled["review_body"].isin(df_clustered["review_body"])]

df_labeled = df_labeled.merge(
    df_clustered[["review_body", "problem_category"]],
    on="review_body",
    how="left"
)

# Problem olmayanları ekle
df_no_problem = df[df["star_rating"] >= 4].copy()
df_no_problem["problem_category"] = "problem_yok"

df_final = pd.concat([df_labeled, df_no_problem], ignore_index=True)

print(df_final.shape)
df_final["problem_category"].value_counts()

(116681, 7)


problem_category
problem_yok           106619
ürün_kalitesi           4216
ürün_dayanıklılığı      2525
performans              1857
içerik_beklenti         1464
Name: count, dtype: int64

In [15]:
df_final.to_csv("data/labeled_data.csv", index=False)
print("Kaydedildi:", df_final.shape)

Kaydedildi: (116681, 7)
